# GPRA Year 3 — Technical Assistance Session Satisfaction Analysis

This notebook processes and summarizes survey data collected after **Technical Assistance (TA) sessions** delivered to grantees under a federal school mental health grant program.

### What this notebook does
1. Loads TA satisfaction survey data from Excel
2. Maps Likert-scale text responses to numeric scores (1–5)
3. Assigns each survey response to its corresponding TA session using submission timestamps
4. Groups survey items into three constructs: **Quality**, **Usefulness**, and **Relevance**
5. Calculates the percentage of respondents rating each construct ≥ 4 ("Agree" or better) per session
6. Exports a session-level summary table for reporting

### Data
Input data are synthetic and anonymized for portfolio purposes. The schema mirrors the structure of real Smartsheet survey exports.

---

## 1. Setup

Load libraries and configure logging.

In [ ]:
import pandas as pd
import numpy as np
import logging
import os
from datetime import timedelta

# ── Logging setup ─────────────────────────────────────────────────────────────
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("gpra3_scoring.log"),
        logging.StreamHandler()
    ]
)
logging.info("Setup complete.")

## 2. Load Data

Read the TA satisfaction survey from Excel.

The `Created` column contains the survey submission timestamp and will be used to match responses to sessions.

In [ ]:
# Update DATA_DIR to point to your local data folder
DATA_DIR = os.path.join(os.getcwd(), "data")
survey_file_path = os.path.join(DATA_DIR, "TA_Satisfaction_Survey.xlsx")

if not os.path.exists(survey_file_path):
    logging.error(f"File not found: {survey_file_path}")
    raise FileNotFoundError(f"File not found: {survey_file_path}")

df = pd.read_excel(survey_file_path, sheet_name=0)

print(f"Loaded {len(df)} survey responses, {df.shape[1]} columns")
logging.info(f"Survey data loaded from {survey_file_path}")

## 3. Recode Likert Responses

Survey items use two different Likert scales (agreement and satisfaction), both mapped to 1–5.
Text responses are replaced with numeric values throughout the DataFrame.

In [ ]:
likert_map = {
    # Agreement scale
    "Strongly Disagree": 1,
    "Disagree": 2,
    "Neither agree nor disagree": 3,
    "Agree": 4,
    "Strongly agree": 5,
    # Satisfaction scale
    "Very Dissatisfied": 1,
    "Dissatisfied": 2,
    "Neither satisfied nor dissatisfied": 3,
    "Satisfied": 4,
    "Very Satisfied": 5
}

df = df.replace(likert_map)
print("Likert recoding complete.")

## 4. Define Survey Item Constructs

Survey items are grouped into three theoretical constructs:

| Construct | Description |
|-----------|-------------|
| **Quality** | Presenter knowledge, clarity, and overall satisfaction |
| **Usefulness** | Applicability, ease of use, and effectiveness of session content |
| **Relevance** | Whether content was new and informative |

In [ ]:
quality = [
    "Presenters Knowledgable", "Presenter communicated clearly",
    "Presenters satisfactory answers", "Presenters Interest",
    "Overall satisfaction", "Content satisfaction"
]
usefulness = [
    "Met expecations", "Strategies applicable", "Strategies doable",
    "Strategies easy to use", "Effective in topic", "Liked session"
]
relevance = ["New content", "Content information"]

all_items = quality + usefulness + relevance
print(f"Total survey items: {len(all_items)} ({len(quality)} quality, {len(usefulness)} usefulness, {len(relevance)} relevance)")

## 5. Parse Timestamps and Assign Sessions

Each survey response is matched to a TA session based on its submission timestamp.

A ±2/+3 day window around each session's anchor date handles late submissions and timezone variation.

In [ ]:
# Parse submission timestamps
df['Created'] = pd.to_datetime(df['Created'], format='%m/%d/%y %I:%M %p', errors='coerce')

n_unparsed = df['Created'].isna().sum()
if n_unparsed > 0:
    logging.warning(f"{n_unparsed} 'Created' timestamps could not be parsed and will be unmatched.")

# Session anchor dates — update these to match your program's schedule
session_dates = {
    "TA Session 1": pd.to_datetime("2024-10-16"),
    "TA Session 2": pd.to_datetime("2024-11-20"),
    "TA Session 3": pd.to_datetime("2025-01-08"),
    "TA Session 4": pd.to_datetime("2025-03-12"),
    "TA Session 5": pd.to_datetime("2025-04-09"),
    "TA Session 6": pd.to_datetime("2025-05-14"),
    "TA Session 7": pd.to_datetime("2025-06-04"),
}


def assign_session(created_date):
    """Match a response timestamp to a session within a ±2/+3 day window."""
    if pd.isna(created_date):
        return "Unmatched"
    for session_name, session_date in session_dates.items():
        if session_date - timedelta(days=2) <= created_date <= session_date + timedelta(days=3):
            return session_name
    return "Unmatched"


df['Session'] = df['Created'].apply(assign_session)

session_counts = df['Session'].value_counts()
print("Responses per session:")
print(session_counts.to_string())

## 6. Summarize by Session

For each session, calculate:
- **Sample size** (number of respondents)
- **Positive response rate** per construct: percentage of all item-respondent combinations scoring ≥ 4

A score of ≥ 4 corresponds to "Agree" or "Satisfied" — the standard threshold for federal TA reporting.

In [ ]:
def get_positive_rate(group, cols):
    """
    Calculate the proportion of item-respondent cells scoring >= 4.
    Returns (count of 4+ scores, total non-null cells).
    """
    # Only include columns that exist in the data
    valid_cols = [c for c in cols if c in group.columns]
    if not valid_cols:
        return 0, 0
    mask = group[valid_cols].apply(pd.to_numeric, errors='coerce').ge(4)
    return int(mask.sum().sum()), int(mask.count().sum())


summary = []

for session, group in df.groupby("Session"):
    if session == "Unmatched":
        continue

    sample_size = len(group)
    q_n, q_total = get_positive_rate(group, quality)
    u_n, u_total = get_positive_rate(group, usefulness)
    r_n, r_total = get_positive_rate(group, relevance)
    o_n, o_total = get_positive_rate(group, all_items)

    summary.append({
        "Session": session,
        "Sample Size": sample_size,
        "n_Overall": o_n,
        "Overall": f"{o_n / o_total:.0%}" if o_total else "NA",
        "n_Quality": q_n,
        "Quality": f"{q_n / q_total:.0%}" if q_total else "NA",
        "n_Usefulness": u_n,
        "Usefulness": f"{u_n / u_total:.0%}" if u_total else "NA",
        "n_Relevance": r_n,
        "Relevance": f"{r_n / r_total:.0%}" if r_total else "NA",
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

## 7. Export Results

In [ ]:
output_dir = os.path.join(os.getcwd(), "output")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "ta_session_summary.csv")
summary_df.to_csv(output_path, index=False)

logging.info(f"Session summary saved to {output_path}")
logging.info("GPRA 3 scoring script completed successfully.")
print(f"Summary saved to {output_path}")